In [ ]:
import os
import pandas as pd
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio
pio.get_chrome()
from pywaffle import Waffle

In [ ]:
# Get base_dir from snakemake variables
if 'snakemake' in globals():
    tsv = snakemake.input[0]
else:
    base_dir = Path("/mnt/cbib/LNClassifier/paper")
    tsv = base_dir / "results/gencode_comparison/v46_vs_v47/gencode.v47.comparison.tsv"

df = pd.read_csv(tsv, sep="\t")

/tmp/ipykernel_534848/3277800397.py:8: DtypeWarning:

Columns (1,2,4,5) have mixed types. Specify dtype option on import or set low_memory=False.



In [ ]:
df

,clean_id,old_full_id,old_seq_id,old_id_version,old_biotype,old_class,new_full_id,new_seq_id,new_id_version,new_biotype,...,common_class_change,pc_to_lncRNA,lncRNA_to_pc,added_with_class,added_pc,added_lncRNA,removed_with_class,removed_pc,removed_lncRNA,no_class
0,ENST00000000233,ENST00000000233.10|ENSG00000004059.11|OTTHUMG0...,ENST00000000233.10,10.0,protein_coding,pc,ENST00000000233.10|ENSG00000004059.11|OTTHUMG0...,ENST00000000233.10,10.0,protein_coding,...,False,False,False,False,False,False,False,False,False,False
1,ENST00000000412,ENST00000000412.8|ENSG00000003056.8|OTTHUMG000...,ENST00000000412.8,8.0,protein_coding,pc,ENST00000000412.8|ENSG00000003056.8|OTTHUMG000...,ENST00000000412.8,8.0,protein_coding,...,False,False,False,False,False,False,False,False,False,False
2,ENST00000000442,ENST00000000442.11|ENSG00000173153.17|OTTHUMG0...,ENST00000000442.11,11.0,protein_coding,pc,ENST00000000442.11|ENSG00000173153.17|OTTHUMG0...,ENST00000000442.11,11.0,protein_coding,...,False,False,False,False,False,False,False,False,False,False
3,ENST00000001008,ENST00000001008.6|ENSG00000004478.8|OTTHUMG000...,ENST00000001008.6,6.0,protein_coding,pc,ENST00000001008.6|ENSG00000004478.8|OTTHUMG000...,ENST00000001008.6,6.0,protein_coding,...,False,False,False,False,False,False,False,False,False,False
4,ENST00000001146,ENST00000001146.7|ENSG00000003137.9|OTTHUMG000...,ENST00000001146.7,7.0,protein_coding,pc,ENST00000001146.7|ENSG00000003137.9|OTTHUMG000...,ENST00000001146.7,7.0,protein_coding,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385872,ENST00000850839,NaN,NaN,NaN,NaN,NaN,ENST00000850839.1|ENSG00000310540.2|OTTHUMG000...,ENST00000850839.1,1.0,lncRNA,...,False,False,False,True,False,True,False,False,False,False
385873,ENST00000850840,NaN,NaN,NaN,NaN,NaN,ENST00000850840.1|ENSG00000292360.2|-|-|LINC03...,ENST00000850840.1,1.0,lncRNA,...,False,False,False,True,False,True,False,False,False,False
385874,ENST00000850841,NaN,NaN,NaN,NaN,NaN,ENST00000850841.1|ENSG00000292360.2|-|-|LINC03...,ENST00000850841.1,1.0,lncRNA,...,False,False,False,True,False,True,False,False,False,False
385875,ENST00000850842,NaN,NaN,NaN,NaN,NaN,ENST00000850842.1|ENSG00000292360.2|-|-|LINC03...,ENST00000850842.1,1.0,lncRNA,...,False,False,False,True,False,True,False,False,False,False


In [ ]:
df.columns

Index(['clean_id', 'old_full_id', 'old_seq_id', 'old_id_version',
       'old_biotype', 'old_class', 'new_full_id', 'new_seq_id',
       'new_id_version', 'new_biotype', 'new_class', 'is_common',
       'transcript_added', 'transcript_removed', 'version_changed',
       'biotype_changed', 'both_have_class', 'same_class', 'class_changed',
       'class_added', 'class_removed', 'common_same_class',
       'common_class_change', 'pc_to_lncRNA', 'lncRNA_to_pc',
       'added_with_class', 'added_pc', 'added_lncRNA', 'removed_with_class',
       'removed_pc', 'removed_lncRNA', 'no_class'],
      dtype='object')

In [ ]:
"""
Class map
0: pc
1: lncRNA
2: no_class
3: no_present
"""
df["source"] = None
df["target"] = None

# Add pc and lncRNA classes
df.loc[df["old_class"] == "pc", "source"] = 0
df.loc[df["old_class"] == "lncRNA", "source"] = 1
df.loc[df["new_class"] == "pc", "target"] = 0
df.loc[df["new_class"] == "lncRNA", "target"] = 1

# Add transcripts added/removed
df.loc[df["transcript_added"], "source"] = 3
df.loc[df["transcript_removed"], "target"] = 3

# The rest gained or lost class
df.loc[df["source"].isna(), "source"] = 2
df.loc[df["target"].isna(), "target"] = 2

sankey_cols = ["source", "target"]

df_sankey = df[sankey_cols].value_counts().reset_index(name="count")
df_sankey["target"] = df_sankey["target"] + 4  # Shift target indices
df_sankey


,source,target,count
0,3,5,131424
1,0,4,111838
2,2,6,82231
3,1,5,59674
4,3,4,331
5,1,7,187
6,3,6,52
7,1,6,43
8,2,4,26
9,1,4,23


In [ ]:
source = df_sankey["source"].tolist()
target = df_sankey["target"].tolist()
value = df_sankey["count"].tolist()

In [ ]:
import colorsys

def lighten_color(hex_color, factor=0.3):
    """
    Lighten a hex color by increasing its lightness.
    
    Parameters:
    -----------
    hex_color : str
        Hex color code (e.g., '#d95f02' or 'd95f02')
    factor : float
        Amount to lighten (0-1). Higher = lighter. Default 0.3
    
    Returns:
    --------
    str
        Lightened hex color code
    """
    # Remove '#' if present
    hex_color = hex_color.lstrip('#')
    
    # Convert hex to RGB (0-1 range)
    r, g, b = tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))
    
    # Convert to HLS
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    
    # Increase lightness
    l = min(1.0, l + (1.0 - l) * factor)
    
    # Convert back to RGB
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    
    # Convert to hex
    return '#{:02x}{:02x}{:02x}'.format(int(r * 255), int(g * 255), int(b * 255))

# Example usage with your current colors
lighter_orange = lighten_color('#d95f02', 0.7)
lighter_purple = lighten_color('#9467bd', 0.7)
lighter_gray = lighten_color('#bdbdbd', 0.7)

print(f"Original: #d95f02 → Lighter: {lighter_orange}")
print(f"Original: #9467bd → Lighter: {lighter_purple}")
print(f"Original: #bdbdbd → Lighter: {lighter_gray}")

Original: #d95f02 → Lighter: #fecda8
Original: #9467bd → Lighter: #ded1eb
Original: #bdbdbd → Lighter: #ebebeb


In [ ]:
from IPython.display import Image, display

# Create link colors based on target node (for visual consistency)
link_colors = []
target_color_map = {4: '#fecda8', 5: "#ded1eb", 6: 'LightSteelBlue', 7: '#ebebeb'}
strong_color_map = {4: '#d95f02', 5: "#9467bd", 6: 'steelblue', 7: '#808080'}
strong_flow_threshold = 1000  # Adjust this threshold based on your data
for tgt, val in zip(target, value):
    if val > strong_flow_threshold:
        link_colors.append(target_color_map[tgt])  # Use lighter color for stronger flows
    else:
        # Use stronger color for smaller flows to maintain visibility
        link_colors.append(strong_color_map[tgt])

# Define node colors (left = old classes, right = new classes)
node_colors = ["#d95f02", "#9467bd", 'steelblue', '#bdbdbd',  # Left side (old)
              "#d95f02", "#9467bd", 'steelblue', '#808080']   # Right side (new)

labels = ['protein-coding', 'lncRNA', 'no class', 'new transcripts',
          'protein-coding', 'lncRNA', 'no class', 'removed transcripts']


x_positions = [0.01, 0.01, 0.01, 0.15555552, 0.99, 0.99, 0.99, 0.5]
y_positions = [0.30, 0.55, 0.05, 0.88, 0.30, 0.65, 0.05, 0.98]

x_positions = [0.01, 0.01, 0.01, 0.25, 0.99, 0.99, 0.99, 0.65]
y_positions = [0.60, 0.35, 0.85, 0.09, 0.60, 0.2, 0.85, 0.99]

# Create the figure with improvements
fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=labels,
        pad=30,
        thickness=50,
        color=node_colors,
        line=dict(color='white', width=2),
        x=x_positions,
        y=y_positions,
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors,  # Color links by source
        hovertemplate='%{source.label} → %{target.label}: %{value:,}<extra></extra>'
    )
)])

######################
# Sankey annotations #
######################
annotation_values = df.groupby("source")["source"].count().to_list() + df.groupby("target")["target"].count().to_list()

def format_count(count):
    if count >= 1_000:
        return f"{count/1_000:.1f} K"
    else:
        return str(count)

# pc, lnc, no_class, new, pc, lnc, no_class, removed
annotation_x = [-0.15, -0.15, -0.15, 0.085, 1.15, 1.15, 1.15, 1.08]
annotation_y = [0.4, 0.65, 0.1, 0.94, 0.4, 0.85, 0.1, -0.03]
annotation_values = [format_count(count) for count in annotation_values]
annotation_align =["right"] * 4 + ["left"] * 4

def annotate_node(ann_pos_x, ann_pos_y, text, text_size=26, align="center"):
    fig.add_annotation(
        x=ann_pos_x,
        y=ann_pos_y,
        text=text,
        showarrow=False,
        font=dict(size=text_size, color='black', family="Arial"),
        align=align,
    )

for ann_x, ann_y, count, align in zip(annotation_x, annotation_y, annotation_values, annotation_align):
    annotate_node(ann_x, ann_y, count, text_size=26, align=align)

# Annotate total version counts
total_old = df["old_seq_id"].notna().sum()
total_new = df["new_seq_id"].notna().sum()
annotate_node(-0.145, 1.22, f"GENCODE v46<br>{format_count(total_old)} transcripts", text_size=30)
annotate_node(1.145, 1.22, f"GENCODE v47<br>{format_count(total_new)} transcripts", text_size=30)


h_cm = 5
w_cm = 8
dpi = 300

width_px = int(w_cm / 2.54 * dpi)
height_px = int(h_cm / 2.54 * dpi)

fig.update_layout(
    height=height_px,
    width=width_px,
    margin=dict(l=110, r=110, t=100, b=20),
    font_family="Arial",
    font_size=26,
)

#png_bytes = fig.to_image(format="png", width=width_px, height=height_px, scale=1)
#display(Image(png_bytes))

DATASET = "gencode.v47.common.cdhit.cv"
FIGURE_DIR = f"/mnt/cbib/LNClassifier/paper/results/{DATASET}/figures"
os.makedirs(FIGURE_DIR, exist_ok=True)
pio.write_image(fig, f"{FIGURE_DIR}/sankey_gencode_versions.pdf", width=width_px, height=height_px, scale=1)

fig.show()

In [ ]:
[1-c for c in [0.60, 0.35, 0.85, 0.12, 0.60, 0.2, 0.85, 0.98]]


[0.4,
 0.65,
 0.15000000000000002,
 0.88,
 0.4,
 0.8,
 0.15000000000000002,
 0.020000000000000018]

In [ ]:
df.groupby("target")["target"].count().reset_index(name="count").set_index("target").reindex([1, 0, 2, 3]).reset_index()

,target,count
0,1,191106
1,0,112218
2,2,82335
3,3,218


In [ ]:
df.groupby("source")["source"].count().reset_index(name="count").set_index("source").reindex([3, 1, 0, 2]).reset_index()

,source,count
0,3,131807
1,1,59927
2,0,111868
3,2,82275


In [ ]:
import plotly.graph_objects as go

# ============================================================================
# IMPROVED GENCODE VERSION COMPARISON SANKEY DIAGRAM
# ============================================================================
# This script creates a publication-quality Sankey plot comparing GENCODE
# versions v46 vs v47 with enhanced styling, positioning, and interactivity

# ============================================================================
# DATA SETUP (replace with your actual data)
# ============================================================================

# Node labels (left = v46, right = v47)
labels = [
    'protein-coding',      # 0 (v46)
    'lncRNA',              # 1 (v46)
    'no class',            # 2 (v46)
    'new transcripts',     # 3 (v46)
    'protein-coding',      # 4 (v47)
    'lncRNA',              # 5 (v47)
    'no class',            # 6 (v47)
    'removed transcripts'  # 7 (v47)
]

# ============================================================================
# COLOR SCHEMES
# ============================================================================

# Node colors - colorblind-friendly palette (Dark2-like)
node_colors = [
    "#1b9e77",  # teal (protein-coding) - v46
    "#d95f02",  # orange (lncRNA) - v46
    "#7570b3",  # purple (no class) - v46
    "#999999",  # gray (new transcripts) - v46
    "#1b9e77",  # teal (protein-coding) - v47
    "#d95f02",  # orange (lncRNA) - v47
    "#7570b3",  # purple (no class) - v47
    "#999999"   # gray (removed transcripts) - v47
]

# Base link colors by source node
source_color_map = {
    0: 'rgba(27, 158, 119, 0.5)',    # teal (protein-coding)
    1: 'rgba(217, 95, 2, 0.5)',      # orange (lncRNA)
    2: 'rgba(117, 112, 179, 0.5)',   # purple (no class)
    3: 'rgba(153, 153, 153, 0.5)'    # gray (new/removed)
}

link_colors = [source_color_map[src] for src in source]

# ============================================================================
# LINK OPACITY BASED ON FLOW MAGNITUDE (for visual hierarchy)
# ============================================================================

min_val = min(value)
max_val = max(value)
link_opacity = [0.35 + 0.65 * (v - min_val) / (max_val - min_val) for v in value]

# Apply opacity to link colors
link_colors_final = []
for color_str, opacity in zip(link_colors, link_opacity):
    rgba_values = color_str.replace('rgba(', '').replace(')', '').split(', ')
    r, g, b = rgba_values[0], rgba_values[1], rgba_values[2]
    link_colors_final.append(f'rgba({r}, {g}, {b}, {opacity})')

# ============================================================================
# NODE POSITIONING (explicit left/right layout)
# ============================================================================

pass

# ============================================================================
# CREATE THE SANKEY FIGURE
# ============================================================================

fig = go.Figure(data=[go.Sankey(
    node=dict(
        label=labels,
        pad=25,          # Vertical space between nodes
        thickness=30,    # Node box height
        color=node_colors,
        line=dict(color='white', width=2),
     # Explicit vertical positioning
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
        color=link_colors_final,
        hovertemplate=(
            '<b>%{source.label}</b> → <b>%{target.label}</b><br>'
            'Count: %{value:,}<extra></extra>'
        )
    )
)])

# ============================================================================
# LAYOUT CONFIGURATION
# ============================================================================

# Calculate total transcripts for summary
total_v46 = sum([value[i] for i, src in enumerate(source) if src in [0, 1, 2]])
total_v47 = sum([value[i] for i, tgt in enumerate(target) if tgt in [4, 5, 6]])

fig.update_layout(
    title={
        'text': (
            '<b>GENCODE version comparison (v46 vs v47)</b>'
            '<br><sub>Transcript reclassification and annotation updates</sub>'
        ),
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'family': 'Arial, sans-serif'}
    },
    font=dict(
        size=13,
        family='Arial, sans-serif',
        color='#333333'
    ),
    height=650,
    width=1100,
    margin=dict(l=100, r=100, t=120, b=50),
    plot_bgcolor='rgba(245, 245, 245, 0.8)',
    paper_bgcolor='white',
    hoverlabel=dict(
        bgcolor='white',
        font_size=12,
        font_family='monospace'
    )
)

# ============================================================================
# ADD ANNOTATIONS (version labels and summary stats)
# ============================================================================

fig.add_annotation(
    text='<b>v46 (reference)</b>',
    x=-0.08, y=0.5,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=12, color='#666666'),
    xanchor='right'
)

fig.add_annotation(
    text='<b>v47 (updated)</b>',
    x=1.08, y=0.5,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=12, color='#666666'),
    xanchor='left'
)

fig.add_annotation(
    text=f'<b>Total v46:</b> {total_v46:,} transcripts',
    x=0.01, y=1.08,
    xref='paper', yref='paper',
    showarrow=False,
    font=dict(size=11, color='#555555'),
    xanchor='left',
    bgcolor='rgba(255, 255, 255, 0.8)',
    bordercolor='#cccccc',
    borderwidth=1,
    borderpad=4
)

# ============================================================================
# DISPLAY AND EXPORT OPTIONS
# ============================================================================

# Display in notebook or browser
fig.show()

# Uncomment below to save as static images (requires: pip install kaleido)
# fig.write_image('gencode_comparison.png', width=1200, height=650, scale=2)
# fig.write_image('gencode_comparison.pdf', width=1200, height=650)

# Save as interactive HTML
# fig.write_html('gencode_comparison_interactive.html')

# ============================================================================
# OPTIONAL: Print summary statistics
# ============================================================================

print("\n" + "="*70)
print("GENCODE v46 vs v47 COMPARISON SUMMARY")
print("="*70)
print(f"\nTotal flow v46 → v47: {sum(value):,} transcripts")
print("\nFlow breakdown:")
for src, tgt, val in zip(source, target, value):
    print(f"  {labels[src]:25} → {labels[tgt]:25} : {val:8,}")
print("\n" + "="*70)



GENCODE v46 vs v47 COMPARISON SUMMARY

Total flow v46 → v47: 385,877 transcripts

Flow breakdown:
  new transcripts           → lncRNA                    :  131,424
  protein-coding            → protein-coding            :  111,838
  no class                  → no class                  :   82,231
  lncRNA                    → lncRNA                    :   59,674
  new transcripts           → protein-coding            :      331
  lncRNA                    → removed transcripts       :      187
  new transcripts           → no class                  :       52
  lncRNA                    → no class                  :       43
  no class                  → protein-coding            :       26
  lncRNA                    → protein-coding            :       23
  protein-coding            → removed transcripts       :       20
  no class                  → removed transcripts       :       11
  protein-coding            → no class                  :        9
  no class                  → 